In [4]:
"""
True:
Unique light edge for every cut ⇒ unique MST

False:
Unique MST ⇒ unique light edge for every cut

If all weights > 0:
cycles are always wasteful ⇒ minimum connected set is a tree

If weights ≤ 0 allowed:
cycles may reduce cost ⇒ optimal structure may not be a tree

Input: MST T, updated edge e = (u,v)
1. Find the unique path P between u and v in T
2. Let C = P ∪ {e}   (this is the cycle)
3. Let f = maximum-weight edge in C (under new weights)
4. If f == e:
       return T (unchanged MST)
   else:
       T = T - {f} + {e}
       return T
=================================================================

A --1-- B
|       |
4       2
|       |
D --3-- C

And one extra diagonal edge:

A --5-- C   (NOT in MST initially)

So edges are:
(A,B) = 1
(B,C) = 2
(C,D) = 3
(D,A) = 4
(A,C) = 5 ← not in MST

Pick smallest edges avoiding cycles:
(A,B) = 1
(B,C) = 2
(C,D) = 3

So MST 
T:
A --1-- B --2-- C --3-- D
Total weight = 6

change a NON-tree edge
Decrease:
(A,C):5→1
Now it becomes very cheap.

Current MST path from A to C:
A --1-- B --2-- C

Cycle formed:
A --1-- B --2-- C
 \           /
  \--1------/

Find max edge in cycle
Cycle weights:
1, 2, 1

Maximum edge = (B,C) = 2

Decide swap
Compare:
new edge (A,C) = 1
removed edge (B,C) = 2

Since:
1<2
replace (B,C) with (A,C)

New MST
Resulting tree:

A --1-- B
 \      |
  1     2
   \    |
     C--3--D

Edges:

(A,B) = 1
(A,C) = 1
(C,D) = 3
Total weight = 5

Before:
MST used path A–B–C
After:
MST uses shortcut A–C instead
"""

'\nTrue:\nUnique light edge for every cut ⇒ unique MST\n\nFalse:\nUnique MST ⇒ unique light edge for every cut\n\nIf all weights > 0:\ncycles are always wasteful ⇒ minimum connected set is a tree\n\nIf weights ≤ 0 allowed:\ncycles may reduce cost ⇒ optimal structure may not be a tree\n\nInput: MST T, updated edge e = (u,v)\n1. Find the unique path P between u and v in T\n2. Let C = P ∪ {e}   (this is the cycle)\n3. Let f = maximum-weight edge in C (under new weights)\n4. If f == e:\n       return T (unchanged MST)\n   else:\n       T = T - {f} + {e}\n       return T\n=================================================================\n\nA --1-- B\n|       |\n4       2\n|       |\nD --3-- C\n\nAnd one extra diagonal edge:\n\nA --5-- C   (NOT in MST initially)\n\nSo edges are:\n(A,B) = 1\n(B,C) = 2\n(C,D) = 3\n(D,A) = 4\n(A,C) = 5 ← not in MST\n\nPick smallest edges avoiding cycles:\n(A,B) = 1\n(B,C) = 2\n(C,D) = 3\n\nSo MST \nT:\nA --1-- B --2-- C --3-- D\nTotal weight = 6\n\nchange a NON

In [6]:
"""
edges: (u, v, weight)

G = {0, 1, 2, 3}
edges = [
    (0, 1, 10),
    (0, 2, 6),
    (0, 3, 5),
    (1, 3, 15),
    (2, 3, 4)
]

mst = mst_kruskal(G, edges)
print(mst)

[(2, 3, 4), (0, 3, 5), (0, 1, 10)]

prevents cycles from being added

MST is not always unique when weights are equal
Different valid MSTs can form depending on:
edge ordering
tie-breaking
"""
def mst_kruskal(G, w):
    parent = {v: v for v in G}
    rank = {v: 0 for v in G}

    def find(v):
        if parent[v] != v:
            parent[v] = find(parent[v])  # Path compression
        return parent[v]

    def union(u, v):
        root_u = find(u)
        root_v = find(v)

        if root_u == root_v:
            return False  # cycle

        # Union by rank
        if rank[root_u] < rank[root_v]:
            parent[root_u] = root_v
        elif rank[root_u] > rank[root_v]:
            parent[root_v] = root_u
        else:
            parent[root_v] = root_u
            rank[root_u] += 1

        return True

    edges = sorted(w, key=lambda x: x[2])

    mst = []

    for u, v, weight in edges:
        if union(u, v):
            mst.append((u, v, weight))

    return mst

In [7]:
"""
# Create vertices
A = Vertex('A')
B = Vertex('B')
C = Vertex('C')
D = Vertex('D')

# Graph structure
G = {
    "V": [A, B, C, D],
    "Adj": {
        A: [B, C],
        B: [A, C, D],
        C: [A, B, D],
        D: [B, C]
    }
}

# Weight function
def w(u, v):
    weights = {
        (A, B): 2, (B, A): 2,
        (A, C): 3, (C, A): 3,
        (B, C): 1, (C, B): 1,
        (B, D): 4, (D, B): 4,
        (C, D): 5, (D, C): 5
    }
    return weights[(u, v)]

mst_prim(G, w, A)

for v in G["V"]:
    print(f"{v} <- {v.p} (weight {v.key})")
"""
import math

class Vertex:
    def __init__(self, name):
        self.name = name
        self.key = math.inf
        self.p = None

    def __repr__(self):
        return self.name

def insert(Q, u):
    Q.append(u)

def extract_min(Q):
    u = min(Q, key=lambda x: x.key)
    Q.remove(u)
    return u

def decrease_key(Q, v, new_key):
    v.key = new_key

def mst_prim(G, w, r):
    for u in G["V"]:
        u.key = math.inf
        u.p = None
    
    r.key = 0

    Q = []
    for u in G["V"]:
        insert(Q, u)

    while Q:
        u = extract_min(Q)

        for v in G["Adj"][u]:
            if v in Q and w(u, v) < v.key:
                v.p = u
                decrease_key(Q, v, w(u, v))

    return G["V"]

In [8]:
"""
using an adjcency matrix

      (1)
   A ------ B
   | \      |
 (4)|  (3)  |(2)
   |    \   |
   D ------ C
       (5)

    A  B  C  D
A [ 0, 1, 3, 4 ]
B [ 1, 0, 2, 0 ]
C [ 3, 2, 0, 5 ]
D [ 4, 0, 5, 0 ]

initialization
key    = [0, ∞, ∞, ∞]
parent = [-, -, -, -]
inMST  = [F, F, F, F]

Step 1: Pick A
Smallest key = A (0)

Include A:
inMST = [T, F, F, F]

Update neighbors of A:
B: key = 1, parent = A
C: key = 3, parent = A
D: key = 4, parent = A
key    = [0, 1, 3, 4]
parent = [-, A, A, A]

Step 2: Pick B
Smallest key = B (1)

Include B:
inMST = [T, T, F, F]

Update neighbors of B:
C: min(3, 2) → update → key = 2, parent = B
key    = [0, 1, 2, 4]
parent = [-, A, B, A]

...

Final MST
Edges (from parent[]):

A — B (1)
B — C (2)
A — D (4)
   A
  / \
(1) (4)
 B     D
 |
(2)
 |
 C
"""
def prim(adj):
    V = len(adj)
    key = [float('inf')] * V
    parent = [-1] * V
    inMST = [False] * V

    key[0] = 0

    for _ in range(V):
        # pick min key vertex not in MST
        u = -1
        for v in range(V):
            if not inMST[v] and (u == -1 or key[v] < key[u]):
                u = v

        inMST[u] = True

        # update neighbors
        for v in range(V):
            if adj[u][v] != 0 and not inMST[v] and adj[u][v] < key[v]:
                key[v] = adj[u][v]
                parent[v] = u

    return parent

In [12]:
"""
Binary heap: O(ElogV)
Fibonacci heap: O(E+VlogV)

kruskal buckets
O(E) - time
"""
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x):
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]

    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx == ry:
            return False
        if self.rank[rx] < self.rank[ry]:
            self.parent[rx] = ry
        elif self.rank[rx] > self.rank[ry]:
            self.parent[ry] = rx
        else:
            self.parent[ry] = rx
            self.rank[rx] += 1
        return True


def kruskal_bucket(V, edges, W):
    buckets = [[] for _ in range(W + 1)]

    for u, v, w in edges:
        buckets[w].append((u, v, w))

    uf = UnionFind(V)
    mst = []

    # process edges in increasing weight
    for w in range(1, W + 1):
        for u, v, w in buckets[w]:
            if uf.union(u, v):
                mst.append((u, v, w))

    return mst

In [18]:
"""
Why Kruskal's algorithm and Prim work:
Always consider all edges globally
Use safe choices (cut/cycle properties) at every step

Kruskal depends on sorting edges → randomness helps a lot
Prim depends on dynamic priorities → randomness helps much less

Kruskal (static world)
All edges known upfront
↓
Sort once (bucket helps)
↓
Scan once

Prim (dynamic world)
Start with one node
↓
Frontier keeps changing
↓
Must repeatedly pick min + update

Kruskal: Sort once → randomness makes sorting linear
Prim: Continuously decide → randomness doesn’t remove updates

Although Kruskal's is faster
Dense graphs favor Prim

Kruskal:
O(ElogE)

Prim (with adjacency matrix):
Prim is asymptotically faster
Kruskal → 𝑂(𝑉^2 log𝑉)
Prim → O(V^2)

Kruskal must sort all edges

Kruskal:
Needs all edges upfront
Pays cost to sort them

Prim:
Grows tree incrementally
Doesn't need global sorting

If edges are expensive to store or stream in, Prim can be better

Graph representation matters
Adjacency matrix → Prim is natural and fast
Edge list → Kruskal is often simpler

Prefer Kruskal when:
Graph is sparse
You already have an edge list
You can use fast sorting (e.g., small weights)
Prefer Prim when:
Graph is dense
You're using an adjacency matrix
You want incremental growth from a start node

Kruskal: "Sort all edges, then pick safe ones"
Prim: "Grow a tree outward from a node"

They solve the same problem with very different philosophies.
"""

'\nWhy Kruskal\'s algorithm and Prim work:\nAlways consider all edges globally\nUse safe choices (cut/cycle properties) at every step\n\nKruskal depends on sorting edges → randomness helps a lot\nPrim depends on dynamic priorities → randomness helps much less\n\nKruskal (static world)\nAll edges known upfront\n↓\nSort once (bucket helps)\n↓\nScan once\n\nPrim (dynamic world)\nStart with one node\n↓\nFrontier keeps changing\n↓\nMust repeatedly pick min + update\n\nKruskal: Sort once → randomness makes sorting linear\nPrim: Continuously decide → randomness doesn’t remove updates\n\nAlthough Kruskal\'s is faster\nDense graphs favor Prim\n\nKruskal:\nO(ElogE)\n\nPrim (with adjacency matrix):\nPrim is asymptotically faster\nKruskal → 𝑂(𝑉^2 log𝑉)\nPrim → O(V^2)\n\nKruskal must sort all edges\n\nKruskal:\nNeeds all edges upfront\nPays cost to sort them\n\nPrim:\nGrows tree incrementally\nDoesn’t need global sorting\n\nIf edges are expensive to store or stream in, Prim can be better\n\nGraph r

In [19]:
"""
Merge vertices connected by "safe" edges
Treat them as a single node

When we "contract" edges in MST-REDUCE, we are saying:
These edges are definitely safe, so we can treat their endpoints as already connected and solve the rest of the problem at a higher level.

merging vertices into a super-node
shrinking the graph
But you are not yet done building the MST

We're locking in obvious MST edges early, then shrinking the problem so the hard part becomes tiny.

You are shrinking the graph while preserving the structure of the optimal MST
The final MST is:
the contracted edges T, plus
the MST of the smaller graph G'

tracking components (union-find)
rebuilding edges
removing duplicates
maintaining mappings (orig, c)


solving MST in stages:
Stage 1: Lock in obvious edges → T
Stage 2: Shrink graph → G'
Stage 3: Solve MST on G'
Final step:
Combine: Final MST = T ∪ (MST of G' mapped back)

A—B—C—D

After MST-REDUCE:
T={AB,BC}
Contract → components: [ABC], [D]

Now:
You still need to connect [ABC] to D
That's what G' + Prim handles
"""
def mst_reduce(V, E, T):
    """
    V: list of vertices
    E: list of edges (u, v, weight, orig, c)
    T: list to accumulate MST edges
    """

    # Step 1: find minimum edge for each vertex
    min_edge = {u: None for u in V}

    for (u, v, w, orig, c) in E:
        if min_edge[u] is None or c < min_edge[u][2]:
            min_edge[u] = (u, v, w, orig, c)
        if min_edge[v] is None or c < min_edge[v][2]:
            min_edge[v] = (v, u, w, orig, c)

    # Step 2: add chosen edges to T
    chosen_edges = set()
    for u in V:
        if min_edge[u]:
            edge = min_edge[u]
            chosen_edges.add((edge[0], edge[1]))
            T.append(edge[3])  # original edge

    # Step 3: find components using union-find
    parent = {u: u for u in V}

    def find(u):
        while parent[u] != u:
            parent[u] = parent[parent[u]]
            u = parent[u]
        return u

    def union(u, v):
        pu, pv = find(u), find(v)
        if pu != pv:
            parent[pu] = pv

    for (u, v) in chosen_edges:
        union(u, v)

    # Step 4: build contracted graph G'
    edge_map = {}  # (comp_u, comp_v) -> best edge

    for (u, v, w, orig, c) in E:
        cu, cv = find(u), find(v)
        if cu == cv:
            continue

        key = (cu, cv) if cu < cv else (cv, cu)

        if key not in edge_map or c < edge_map[key][2]:
            edge_map[key] = (cu, cv, w, orig, c)

    # new vertices
    V_new = list(set(find(u) for u in V))
    E_new = list(edge_map.values())

    return V_new, E_new, T

In [20]:
"""
Stop shrinking right when the graph becomes just small enough that Prim is cheap.

That happens exactly at:
k=loglogV

Preprocessing helps when:
you can compress structure faster than Prim can exploit heaps

But once edges get moderately dense,
the overhead of repeated reductions outweighs savings.
"""

'\nStop shrinking right when the graph becomes just small enough that Prim is cheap.\n\nThat happens exactly at:\nk=loglogV\n\nPreprocessing helps when:\nyou can compress structure faster than Prim can exploit heaps\n\nBut once edges get moderately dense,\nthe overhead of repeated reductions outweighs savings.\n'

In [21]:
"""
Start with all edges
Remove edges in decreasing weight order
Only remove if graph stays connected
Is it an MST?
Yes always produces an MST
"""
def maybe_mst_a(V, E):
    # Sort edges in decreasing order of weight
    edges = sorted(E, key=lambda x: x[2], reverse=True)
    
    T = set(E)  # start with all edges

    def is_connected(V, edges):
        if not V:
            return True
        
        adj = {v: [] for v in V}
        for u, v, _ in edges:
            adj[u].append(v)
            adj[v].append(u)

        visited = set()
        stack = [next(iter(V))]

        while stack:
            node = stack.pop()
            if node not in visited:
                visited.add(node)
                stack.extend(adj[node])

        return len(visited) == len(V)

    for e in edges:
        temp = T - {e}
        if is_connected(V, temp):
            T = temp

    return T

In [24]:
"""
Add edges in arbitrary order
Only if they don’t create a cycle
Is it an MST?
No

Why it fails
It ignores the cut property:
You must always pick the lightest edge across a cut
Arbitrary order breaks that guarantee
"""
def maybe_mst_b(V, E):
    T = set()

    parent = {v: v for v in V}

    def find(u):
        if parent[u] != u:
            parent[u] = find(parent[u])
        return parent[u]

    def union(u, v):
        parent[find(u)] = find(v)

    for u, v, w in E:  # arbitrary order
        if find(u) != find(v):  # no cycle
            T.add((u, v, w))
            union(u, v)

    return T

In [25]:
"""
Add edges one by one
If a cycle forms:
remove the maximum-weight edge in that cycle
Is it an MST?
Yes always produces an MST

Uses the cycle property directly:
In any cycle, the maximum-weight edge is never in an MST

whenever a cycle appears
removing the heaviest edge is always safe
"""
def maybe_mst_c(V, E):
    T = set()

    def find_cycle(T, new_edge):
        # simple DFS to detect cycle and return it
        from collections import defaultdict
        
        adj = defaultdict(list)
        for u, v, w in T:
            adj[u].append((v, w))
            adj[v].append((u, w))

        u, v, w = new_edge

        # find path from u to v
        stack = [(u, None, [])]
        visited = set()

        while stack:
            node, parent, path = stack.pop()
            if node == v:
                return path + [(u, v, w)]
            
            if node in visited:
                continue
            visited.add(node)

            for nei, wt in adj[node]:
                if nei != parent:
                    stack.append((nei, node, path + [(node, nei, wt)]))
        
        return None  # no cycle

    for e in E:
        T.add(e)
        cycle = find_cycle(T, e)
        
        if cycle:
            # remove max-weight edge in cycle
            max_edge = max(cycle, key=lambda x: x[2])
            if max_edge in T:
                T.remove(max_edge)

    return T

In [26]:
"""
    (A)
   /   \
  4     7
 /       \
(B)---2---(C)
  \       /
   6     5
    \   /
     (D)

V = {"A", "B", "C", "D"}

E = [
    ("A", "B", 4),
    ("A", "C", 7),
    ("B", "C", 2),
    ("B", "D", 6),
    ("C", "D", 5),
]

is_bottleneck_at_most_b(V, E, 5)

Edges kept (≤ 5):
AB = 4
BC = 2
CD = 5

Graph becomes:
(A)--4--(B)--2--(C)--5--(D)
Connected → returns True

is_bottleneck_at_most_b(V, E, 4)
Edges kept (≤ 4):
AB = 4
BC = 2

Graph becomes:
(A)--4--(B)--2--(C)    (D isolated)
Not connected → returns False

Smallest b that works = 5
So bottleneck value = 5

If I forbid all edges heavier than b, can I still connect the graph?
"""
def is_bottleneck_at_most_b(V, E, b):
    adj = {v: [] for v in V}
    
    for u, v, w in E:
        if w <= b:
            adj[u].append(v)
            adj[v].append(u)

    # DFS to check connectivity
    visited = set()
    stack = [next(iter(V))]

    while stack:
        node = stack.pop()
        if node not in visited:
            visited.add(node)
            stack.extend(adj[node])

    return len(visited) == len(V)